# Transferability: pHash attack → PDQ / NeuralHash
Tests whether adversarial examples crafted for pHash also fool PDQ and NeuralHash.

In [ ]:
%matplotlib inline

import sys
from pathlib import Path

REPO = Path('..').resolve()
sys.path.insert(0, str(REPO))
sys.path.insert(0, str(REPO / 'gigaevo-core'))

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import importlib.util

from evohash.dataset import load_image_pairs
from evohash.phf.phash import PHashWrapper
from evohash.phf.pdq import PDQWrapper
from evohash.phf.neuralhash import NeuralHashWrapper

In [ ]:
# ── Load data and hash functions ──────────────────────────────────────────────
import evohash
data_dir = Path(evohash.__file__).parent.parent / 'data' / 'imagenet_val'

N_PAIRS = 100

phash = PHashWrapper()
pdq   = PDQWrapper()
nh    = NeuralHashWrapper()

pairs = load_image_pairs(data_dir, n_pairs=N_PAIRS)
sources = [p[0] for p in pairs]
targets = [p[1] for p in pairs]

# Precompute target hashes for all three functions
target_hashes_phash = [phash.compute(img) for img in targets]
target_hashes_pdq   = [pdq.compute(img)   for img in targets]
target_hashes_nh    = [nh.compute(img)     for img in targets]

print(f'Loaded {N_PAIRS} pairs')
print(f'pHash threshold={phash.threshold}, PDQ threshold={pdq.threshold}, NeuralHash threshold={nh.threshold}')

In [ ]:
# ── Run pHash best attack ─────────────────────────────────────────────────────
attack_path = REPO / 'problems' / 'phash' / 'best_attack.py'
spec = importlib.util.spec_from_file_location('best_attack', attack_path)
mod  = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)

context = {
    'hash_fn':       phash,
    'threshold':     phash.threshold,
    'source_images': sources,
    'target_hashes': target_hashes_phash,
    'target_images': targets,
}

print('Running pHash attack on', N_PAIRS, 'pairs...')
result   = mod.entrypoint(context)
attacked = result['attacked_images']
print('Done.')

In [ ]:
# ── Evaluate all three hash functions on attacked images ──────────────────────
results = {h: {'success': [], 'dist': [], 'l2': []} for h in ['pHash', 'PDQ', 'NeuralHash']}

hash_fns = [
    ('pHash',      phash, target_hashes_phash),
    ('PDQ',        pdq,   target_hashes_pdq),
    ('NeuralHash', nh,    target_hashes_nh),
]

for i, (src, atk) in enumerate(zip(sources, attacked)):
    src_arr = np.array(src).astype(float)
    atk_arr = np.array(atk).astype(float)
    l2 = np.linalg.norm((atk_arr - src_arr).flatten()) / np.sqrt(src_arr.size)

    for name, hfn, tgt_hashes in hash_fns:
        h_atk = hfn.compute(atk)
        dist  = hfn.distance(h_atk, tgt_hashes[i])
        success = dist <= hfn.threshold
        results[name]['success'].append(success)
        results[name]['dist'].append(dist)
        results[name]['l2'].append(l2)

print(f"{'Hash':<12} {'ASR':>6} {'Mean dist':>10} {'Mean L2':>8}")
print('-' * 40)
for name, hfn, _ in hash_fns:
    r = results[name]
    asr  = np.mean(r['success'])
    mdist = np.mean(r['dist'])
    ml2   = np.mean(r['l2'])
    print(f"{name:<12} {asr:>6.3f} {mdist:>10.2f} {ml2:>8.2f}")

In [ ]:
# ── All 100 images: source | attacked | target  with per-hash color badges ────
HASH_COLORS = {
    'pHash':      '#2196F3',   # blue
    'PDQ':        '#FF9800',   # orange
    'NeuralHash': '#9C27B0',   # purple
}

def draw_image_grid(indices, suptitle, sort_key=None):
    """Draw source | attacked | target for each index with hash badges."""
    if sort_key:
        indices = sorted(indices, key=sort_key)
    n = len(indices)
    fig, axes = plt.subplots(n, 3, figsize=(10, 3.2 * n))
    if n == 1:
        axes = axes[np.newaxis, :]
    fig.suptitle(suptitle, fontsize=13, y=1.001, fontweight='bold')
    axes[0, 0].set_title('Source',   fontsize=10)
    axes[0, 1].set_title('Attacked', fontsize=10)
    axes[0, 2].set_title('Target',   fontsize=10)

    for row, i in enumerate(indices):
        for col, img in enumerate([sources[i], attacked[i], targets[i]]):
            axes[row, col].imshow(img)
            axes[row, col].axis('off')

        l2_val = results['pHash']['l2'][i]

        # ── 3 colored badges stacked on attacked image ──
        badge_y = 0.01
        for name, hfn, _ in hash_fns:
            ok     = results[name]['success'][i]
            dist   = results[name]['dist'][i]
            icon   = '✓' if ok else '✗'
            color  = HASH_COLORS[name]
            text   = f'{icon} {name}  {dist:.0f} / {hfn.threshold}'
            axes[row, 1].text(
                0.5, badge_y, text,
                transform=axes[row, 1].transAxes,
                fontsize=7, color='white', ha='center', va='bottom',
                bbox=dict(facecolor=color, alpha=0.85, pad=2, boxstyle='round,pad=0.3')
            )
            badge_y += 0.095

        # L2 label top-right on attacked
        axes[row, 1].text(
            0.98, 0.98, f'L2={l2_val:.1f}',
            transform=axes[row, 1].transAxes,
            fontsize=8, color='white', ha='right', va='top',
            bbox=dict(facecolor='black', alpha=0.6, pad=2)
        )

        # Pair number on source
        axes[row, 0].text(
            0.02, 0.98, f'#{i}',
            transform=axes[row, 0].transAxes,
            fontsize=8, color='white', ha='left', va='top',
            bbox=dict(facecolor='black', alpha=0.5, pad=2)
        )

    plt.tight_layout()
    plt.show()

# All 100 pairs
draw_image_grid(
    list(range(N_PAIRS)),
    suptitle='All 100 pairs — pHash attack  (badges: ✓/✗ per hash, dist / threshold)',
)

In [ ]:
# ── Top-10 best and worst for each hash ───────────────────────────────────────
for name, hfn, _ in hash_fns:
    succ_idx = [i for i in range(N_PAIRS) if     results[name]['success'][i]]
    fail_idx = [i for i in range(N_PAIRS) if not results[name]['success'][i]]

    # best = successful, sorted by dist ascending (closest to 0)
    top10  = sorted(succ_idx, key=lambda i: results[name]['dist'][i])[:10]
    # worst = failed, sorted by dist descending (furthest from threshold)
    bot10  = sorted(fail_idx, key=lambda i: results[name]['dist'][i], reverse=True)[:10]

    # pad with high-L2 successes if not enough failures
    if len(bot10) < 10:
        extra = sorted(succ_idx, key=lambda i: results[name]['dist'][i], reverse=True)
        for idx in extra:
            if idx not in bot10 and len(bot10) < 10:
                bot10.append(idx)

    if top10:
        draw_image_grid(
            top10,
            suptitle=f'Top-10 BEST for {name}  (threshold={hfn.threshold}, sorted by dist ↑)',
        )
    if bot10:
        draw_image_grid(
            bot10,
            suptitle=f'Top-10 WORST for {name}  (threshold={hfn.threshold}, sorted by dist ↓)',
        )

In [ ]:
# ── Summary bar chart ─────────────────────────────────────────────────────────
names  = ['pHash', 'PDQ', 'NeuralHash']
asrs   = [np.mean(results[n]['success']) for n in names]
colors = [HASH_COLORS[n] for n in names]
thresh = [phash.threshold, pdq.threshold, nh.threshold]

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(
    [f'{n}\n(thr={t})' for n, t in zip(names, thresh)],
    asrs, color=colors, edgecolor='white', width=0.5
)
ax.set_ylim(0, 1.15)
ax.set_ylabel('ASR (Attack Success Rate)')
ax.set_title('Transferability: pHash attack on all three hashes', fontweight='bold')
for bar, v in zip(bars, asrs):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.03,
            f'{v:.2f}', ha='center', fontsize=13, fontweight='bold')
ax.axhline(1.0, color='gray', linestyle='--', alpha=0.4, label='ASR=1.0')
ax.legend()
plt.tight_layout()
plt.savefig(REPO / 'notebooks' / 'eval_transferability.png', dpi=100, bbox_inches='tight')
plt.show()